# 9. AIND Ephys QC Collector

Build an `AINDEPhysQCCollectorScanConfig`, expand it with `GridScanGenerationTask`, and run the `AINDEPhysQCCollectorTask` for each coordinate. The task itself clones [aind-ephys-qc-collector](https://github.com/AllenNeuralDynamics/aind-ephys-qc-collector) on first use, seeds its `data/` with the per-recording `quality_control_<name>.json` + `quality_control_<name>/` figure folders from notebook 08, runs `code/run_capsule.py`, and copies the aggregated `quality_control.json` + flat `quality_control/<probe>/` figure tree into the single config's `coordinate_output_root`.

## Imports and prerequisites

In [1]:
import json
import shutil
import subprocess
import sys
from pathlib import Path

import obi_one as obi

In [2]:
subprocess.run(
    ["uv", "pip", "install", "--python", sys.executable, "aind-data-schema"],
    check=True,
)

Using Python 3.12.9 environment at: /Users/james/Documents/obi/code/obi-main/obi-one/.venv
Resolved 14 packages in 10ms
Uninstalled 2 packages in 14ms
Installed 2 packages in 4ms
 - pydantic==2.12.5
 + pydantic==2.11.10
 - pydantic-core==2.41.5
 + pydantic-core==2.33.2


CompletedProcess(args=['uv', 'pip', 'install', '--python', '/Users/james/Documents/obi/code/obi-main/obi-one/.venv/bin/python', 'aind-data-schema'], returncode=0)

## Build the scan config

In [3]:
qc_output_path = (Path.cwd() / "output/08_qc_results").resolve()
assert qc_output_path.exists(), (
    f"{qc_output_path} not found. Run 08_aind_ephys_processing_qc.ipynb first."
)

scan_config = obi.AINDEPhysQCCollectorScanConfig(
    initialize=obi.AINDEPhysQCCollectorScanConfig.Initialize(
        qc_output_path=qc_output_path,
    ),
)

## Generate the grid scan and run the QC-collector task

In [4]:
grid_scan = obi.GridScanGenerationTask(
    form=scan_config,
    output_root="../../../obi-output/aind_ephys_qc_collector/grid_scan",
)
grid_scan.execute()

obi.run_tasks_for_generated_scan(grid_scan)

[2026-04-29 17:23:31,168] INFO: Cloning https://github.com/AllenNeuralDynamics/aind-ephys-qc-collector.git -> /tmp/aind-ephys-qc-collector


Cloning into '/tmp/aind-ephys-qc-collector'...


[2026-04-29 17:23:31,928] INFO: Seeded 1 QC recording(s) into /tmp/aind-ephys-qc-collector/data


[2026-04-29 17:23:31,928] INFO: Running python -u run_capsule.py


[2026-04-29 17:23:32,336] INFO: 
EPHYS QC COLLECTION
Found 1 quality control files
All recording segments are the same. Dropping recording segment from recording name.
	Collected 5 metrics for 0 tags and 1 streams.
	Tags allowed to fail: []
EPHYS QC COLLECTION time: 0.0s



[PosixPath('../../../obi-output/aind_ephys_qc_collector/grid_scan/AINDEPhysQCCollectorSingleConfig')]

## Copy the per-coordinate QC-collected results next to the notebook

In [5]:
local_results_dir = Path.cwd() / "output/09_qc_collected_results"
if local_results_dir.exists():
    shutil.rmtree(local_results_dir)
local_results_dir.mkdir(parents=True, exist_ok=True)

for single_config in grid_scan.single_configs:
    coord_dir = Path(single_config.coordinate_output_root)
    dest = local_results_dir / str(single_config.idx)
    dest.mkdir(parents=True, exist_ok=True)
    for entry in coord_dir.iterdir():
        target = dest / entry.name
        if target.exists():
            shutil.rmtree(target) if target.is_dir() else target.unlink()
        if entry.is_dir():
            shutil.copytree(entry, target)
        else:
            shutil.copy2(entry, target)

first = local_results_dir / "0"
if first.exists():
    for entry in first.iterdir():
        target = local_results_dir / entry.name
        if target.exists():
            shutil.rmtree(target) if target.is_dir() else target.unlink()
        if entry.is_dir():
            shutil.copytree(entry, target)
        else:
            shutil.copy2(entry, target)

for p in sorted(local_results_dir.iterdir()):
    print(" ", p.name)

  0
  obi_one_coordinate.json
  quality_control
  quality_control.json


## Inspect the aggregated quality_control.json

In [6]:
qc = json.loads((local_results_dir / "quality_control.json").read_text())
print("schema_version:", qc.get("schema_version"))
print("default_grouping:", qc.get("default_grouping"))
print("allow_tag_failures:", qc.get("allow_tag_failures"))
print(f"Total metrics: {len(qc.get('metrics', []))}")
for m in qc.get("metrics", []):
    print(f"  {m.get('stage'):>10}  {m.get('name'):40}  reference={m.get('reference')}")

schema_version: 2.4.1
default_grouping: ['probe', 'stage']
allow_tag_failures: []
Total metrics: 5
    Raw data  Raw data block0_None                      reference=quality_control/block0_None/traces_raw.png
    Raw data  PSD block0_None                           reference=quality_control/block0_None/psd.png
    Raw data  RMS block0_None                           reference=quality_control/block0_None/rms.png
  Processing  Unit Metrics Yield - block0_None          reference=quality_control/block0_None/unit_yield.png
  Processing  Firing rate - block0_None                 reference=quality_control/block0_None/firing_rate.png
